# Study 916 — Withholding Drag — the teardown

The measurement identity and its validation, the resolution floor, the income-yield gap with HAC *t* and block-bootstrap CIs, the fee decomposition, the era cut, the weight sweep, the labelled withholding inference, the excess-of-cash total-return race with a cost sweep, and the live synthetic control. Every real number is frozen from `docs/results.md` (fingerprint `ca173c825c7c`, as-of 2026-06-30).

In [1]:
R = {'start': '2007-07-30', 'end': '2026-06-30', 'n_days': 4760, 'fp': 'ca173c825c7c', 'ruler': [('VEA', 302.3, 302.2, 0.1, 66), ('IEFA', 291.3, 291.7, -0.44, 29), ('EFA', 269.3, 269.2, 0.04, 47), ('VXUS', 296.7, 297.2, -0.42, 57), ('EWJ', 147.6, 147.2, 0.38, 50), ('EWG', 232.7, 232.5, 0.16, 42), ('EWU', 373.5, 372.9, 0.6, 56)], 'cal_n': 3437, 'cal_efa': 300.8, 'cal_iefa': 291.3, 'cal_gap': -9.5, 'cal_t': -1.05, 'cal_lo': -28.8, 'cal_hi': 6.5, 'cal_expected': 26.0, 'cal_resid': -35.5, 'head': [('VEA', 4760, 302.4, 254.9, -47.5, -1.07, -0.5, -0.01, -69.3, 69.0), ('IEFA', 3436, 291.4, 257.8, -33.6, -2.77, 9.4, 0.78, -12.5, 30.6), ('EFA', 6244, 269.3, 222.9, -46.4, -4.11, -29.4, -2.61, -49.3, -10.0), ('VXUS', 3875, 296.8, 256.0, -40.8, -0.89, 4.2, 0.09, -56.3, 68.4)], 'fee_gap': 47.0, 'n_years': 18, 'n_negative_years': 17, 'worst_year': 2013, 'worst_gap': -114.6, 'flip_year': 2025, 'flip_gap': 28.0, 'era_e_n': 2375, 'era_e_gap': -59.2, 'era_e_t': -0.85, 'era_e_adj': -12.2, 'era_l_n': 2384, 'era_l_gap': -35.8, 'era_l_t': -0.62, 'era_l_adj': 11.2, 'wsweep': [('EAFE-ish 50/30/20', 254.9, -47.5, -0.5, -1.07), ('equal 1/3 each', 271.5, -30.9, 16.1, -0.65), ('Japan-heavy 70/20/10', 226.5, -75.9, -28.9, -1.82), ('UK-heavy 30/50/20', 298.1, -4.3, 42.7, -0.09)], 'wsweep_span': 71.6, 'fsweep': [(35.0, -12.5, -0.28, -81.3, 57.0), (40.0, -7.5, -0.17, -76.3, 62.0), (44.0, -3.5, -0.08, -72.3, 66.0), (47.0, -0.5, -0.01, -69.3, 69.0), (51.0, 3.5, 0.08, -65.3, 73.0), (55.0, 7.5, 0.17, -61.3, 77.0)], 'net_yield': 302.0, 'infer': [(0.05, 318, 16), (0.1, 336, 34), (0.12, 344, 41), (0.15, 356, 53), (0.2, 378, 76), (0.25, 403, 101)], 'central_w': 0.12, 'central_drag': 41, 'race': [(0.0, 0.288, 0.251, -0.037, -102, -1.05), (5.0, 0.288, 0.25, -0.038, -103, -1.06), (25.0, 0.288, 0.249, -0.04, -107, -1.09)], 'ci_vea_lo': -0.091, 'ci_vea_hi': 0.702, 'ci_blend_lo': -0.132, 'ci_blend_hi': 0.652, 'syn_planted_true': 25.6, 'syn_planted_meas': 25.4, 'syn_planted_t': 9.17, 'syn_null_meas': 0.1, 'syn_null_lo': -0.1, 'syn_null_hi': 0.4}

## The identity

For split-adjusted legs `TR` (dividend-adjusted) and `PX` (not),

    d_t = TR_t/TR_{t-1} - PX_t/PX_{t-1}

is 0 off ex-date and `D_t / PX_{t-1}` on it. Summed over 252 sessions it is the realised **net** distribution yield: net of foreign withholding *and* of the fund's expenses, because both come out before the cash is declared.

> 💡 **In plain words:** the total-return line includes the dividends, the price line does not; the gap between them is the cash you were paid.

Validation against the funds' declared per-share cash distributions:

In [2]:
print(f"{'fund':6s}{'differenced':>13s}{'declared':>12s}{'residual':>11s}{'ex-dates':>10s}")
for tk, diff, cash, resid, nex in R['ruler']:
    print(f'{tk:6s}{diff:11.1f}bp{cash:10.1f}bp{resid:+10.2f}bp{nex:10d}')
print('\nmax |residual| = %.2f bp -> no systematic measurement bias'
      % max(abs(r[3]) for r in R['ruler']))

fund    differenced    declared   residual  ex-dates
VEA         302.3bp     302.2bp     +0.10bp        66
IEFA        291.3bp     291.7bp     -0.44bp        29
EFA         269.3bp     269.2bp     +0.04bp        47
VXUS        296.7bp     297.2bp     -0.42bp        57
EWJ         147.6bp     147.2bp     +0.38bp        50
EWG         232.7bp     232.5bp     +0.16bp        42
EWU         373.5bp     372.9bp     +0.60bp        56

max |residual| = 0.60 bp -> no systematic measurement bias


## The resolution floor — EFA vs IEFA

Same issuer, near-identical market, a **known** 26 bp fee gap (EFA 33 bp, IEFA 7 bp). If the estimator can resolve tens of basis points of *fund-level* income difference, it should recover that. It does not — IEFA's index includes small caps and EFA's does not, and composition swamps the fee signal.

> 💡 **In plain words:** even for two near-twins, the honest error bar is several tens of basis points — as large as the effect we are hunting.

In [3]:
print(f"EFA {R['cal_efa']:.1f} bp/yr vs IEFA {R['cal_iefa']:.1f} bp/yr  (n={R['cal_n']:,})")
print(f"measured gap {R['cal_gap']:+.1f} bp   HAC t {R['cal_t']:+.2f}   "
      f"95% CI [{R['cal_lo']:+.1f}, {R['cal_hi']:+.1f}]")
print(f"fee gap alone predicts {R['cal_expected']:+.1f} bp -> residual {R['cal_resid']:+.1f} bp")
print('\n-> the +26 bp truth sits OUTSIDE the CI. Resolution floor ~ tens of bp.')

EFA 300.8 bp/yr vs IEFA 291.3 bp/yr  (n=3,437)
measured gap -9.5 bp   HAC t -1.05   95% CI [-28.8, +6.5]
fee gap alone predicts +26.0 bp -> residual -35.5 bp

-> the +26 bp truth sits OUTSIDE the CI. Resolution floor ~ tens of bp.


## The headline gap — broad fund minus single-country blend

Blend = EWJ/EWU/EWG 50/30/20, monthly rebalance, weights known at month-end *t* and traded at *t+1* (the study's single execution lag). Raw gap first, then with the 47 bp fee difference added back — a constant shift, so the CI width is unchanged and the HAC *t* is simply re-centred.

In [4]:
hdr = f"{'fund':6s}{'n':>7s}{'fund':>8s}{'blend':>8s}{'gap':>9s}{'t':>7s}{'fee-adj':>10s}{'t':>7s}{'CI(fee-adj)':>20s}"
print(hdr)
for tk, n, f, b, gap, t, adj, tadj, lo, hi in R['head']:
    print(f'{tk:6s}{n:7,d}{f:8.1f}{b:8.1f}{gap:9.1f}{t:+7.2f}{adj:10.1f}{tadj:+7.2f}'
          f'   [{lo:+.1f}, {hi:+.1f}]')
print('\nEvery raw gap is NEGATIVE -- the broad funds distribute MORE than the')
print('blend, the wrong sign for a withholding leak. The only |t|>2 survivor')
print('after fees (EFA, t=-2.61) still points the wrong way and starts in 2001.')

fund        n    fund   blend      gap      t   fee-adj      t         CI(fee-adj)
VEA     4,760   302.4   254.9    -47.5  -1.07      -0.5  -0.01   [-69.3, +69.0]
IEFA    3,436   291.4   257.8    -33.6  -2.77       9.4  +0.78   [-12.5, +30.6]
EFA     6,244   269.3   222.9    -46.4  -4.11     -29.4  -2.61   [-49.3, -10.0]
VXUS    3,875   296.8   256.0    -40.8  -0.89       4.2  +0.09   [-56.3, +68.4]

Every raw gap is NEGATIVE -- the broad funds distribute MORE than the
blend, the wrong sign for a withholding leak. The only |t|>2 survivor
after fees (EFA, t=-2.61) still points the wrong way and starts in 2001.


## Era cut and weight sweep

Split at 2017-01-01. The raw gap is insignificant in both halves and the fee-adjusted gap **changes sign** (-12.2 → +11.2 bp). The weight sweep is the more damning of the two: the composition assumption moves the estimate by 72 bp/yr, more than any plausible tax effect.

> 💡 **In plain words:** you get whatever answer your choice of benchmark countries hands you.

In [5]:
print(f"2007-2016 (n={R['era_e_n']:,}): gap {R['era_e_gap']:+.1f} (t {R['era_e_t']:+.2f})  fee-adj {R['era_e_adj']:+.1f}")
print(f"2017-2026 (n={R['era_l_n']:,}): gap {R['era_l_gap']:+.1f} (t {R['era_l_t']:+.2f})  fee-adj {R['era_l_adj']:+.1f}  <- sign flip")
print()
for name, b, gap, adj, t in R['wsweep']:
    print(f'{name:24s} blend {b:6.1f}  gap {gap:+7.1f}  fee-adj {adj:+7.1f}  (t {t:+.2f})')
print(f'\nfee-adjusted span across weight sets: {R["wsweep_span"]:.0f} bp/yr')

2007-2016 (n=2,375): gap -59.2 (t -0.85)  fee-adj -12.2
2017-2026 (n=2,384): gap -35.8 (t -0.62)  fee-adj +11.2  <- sign flip

EAFE-ish 50/30/20        blend  254.9  gap   -47.5  fee-adj    -0.5  (t -1.07)
equal 1/3 each           blend  271.5  gap   -30.9  fee-adj   +16.1  (t -0.65)
Japan-heavy 70/20/10     blend  226.5  gap   -75.9  fee-adj   -28.9  (t -1.82)
UK-heavy 30/50/20        blend  298.1  gap    -4.3  fee-adj   +42.7  (t -0.09)

fee-adjusted span across weight sets: 72 bp/yr


## The third assumption — expense ratios

The fee add-back uses **today's** (2026) fact-sheet fees on a 19-year window, and ETF fees fell over it: VEA was well above 10 bp at its 2007 launch against 3 bp now, while the iShares single-country funds barely moved. The time-averaged fee gap is therefore *smaller* than 47 bp — so the headline fee-adjusted gap is an **upper bound**, i.e. it already flatters the withholding hypothesis (which needs a *positive* gap).

> 💡 **In plain words:** we gave the theory the most generous fee assumption available and it still did not clear zero.

In [6]:
print(f"{'fee gap':>8s}{'fee-adj gap':>13s}{'HAC t':>8s}{'95% CI':>22s}")
for f, adj, t, lo, hi in R['fsweep']:
    mark = '  <- headline' if abs(f - 47.0) < 1e-9 else ''
    print(f'{f:7.1f}b{adj:+13.1f}{t:+8.2f}   [{lo:+.1f}, {hi:+.1f}]{mark}')
span = R['fsweep'][-1][1] - R['fsweep'][0][1]
print(f'\nfee-adjusted gap spans {span:.0f} bp across the fee grid; max |t| = '
      f"{max(abs(r[2]) for r in R['fsweep']):.2f}. No fee assumption rescues "
      'the sign or the significance.')

 fee gap  fee-adj gap   HAC t                95% CI
   35.0b        -12.5   -0.28   [-81.3, +57.0]
   40.0b         -7.5   -0.17   [-76.3, +62.0]
   44.0b         -3.5   -0.08   [-72.3, +66.0]
   47.0b         -0.5   -0.01   [-69.3, +69.0]  <- headline
   51.0b         +3.5   +0.08   [-65.3, +73.0]
   55.0b         +7.5   +0.17   [-61.3, +77.0]

fee-adjusted gap spans 20 bp across the fee grid; max |t| = 0.28. No fee assumption rescues the sign or the significance.


## INFERENCE — not a measurement

`gross = net / (1 − w)`, `drag = net · w / (1 − w)`. The tape identifies `net` = 302 bp/yr and nothing else, so `w` is an **assumption** (Japan 15% and Germany 15% by treaty, the UK 0% because it levies no dividend withholding tax at all) and the answer is a sweep, never a point.

> 💡 **In plain words:** we can tell you exactly what landed in your account and only guess what was skimmed on the way.

In [7]:
print(f"measured net distribution yield: {R['net_yield']:.0f} bp/yr (VEA)")
for w, gross, drag in R['infer']:
    mark = '  <- central ASSUMPTION' if abs(w - R['central_w']) < 1e-9 else ''
    print(f'  w={w:5.0%}  implied gross {gross:5.0f} bp  implied drag {drag:5.0f} bp/yr{mark}')
lo = R['infer'][0][2]; hi = R['infer'][-1][2]
print(f'\nhonest range {lo}-{hi} bp/yr ({hi/lo:.1f}x), driven purely by w.')
print('A US taxable holder recovers much of this via the foreign tax credit;')
print('an IRA/401(k) holder does not. Off-tape, not modelled.')

measured net distribution yield: 302 bp/yr (VEA)
  w=   5%  implied gross   318 bp  implied drag    16 bp/yr
  w=  10%  implied gross   336 bp  implied drag    34 bp/yr
  w=  12%  implied gross   344 bp  implied drag    41 bp/yr  <- central ASSUMPTION
  w=  15%  implied gross   356 bp  implied drag    53 bp/yr
  w=  20%  implied gross   378 bp  implied drag    76 bp/yr
  w=  25%  implied gross   403 bp  implied drag   101 bp/yr

honest range 16-101 bp/yr (6.3x), driven purely by w.
A US taxable holder recovers much of this via the foreign tax credit;
an IRA/401(k) holder does not. Off-tape, not modelled.


## Is it bankable? Excess-of-cash race, VEA vs the blend

Both legs excess of BIL's own total return. Costs are charged one-way × NAV on the blend's monthly rebalance turnover only; VEA is buy-and-hold. No short leg anywhere, so no borrow applies.

In [8]:
print(f"{'cost':>6s}{'VEA exSh':>11s}{'blend exSh':>13s}{'adv':>9s}{'ret adv':>11s}{'HAC t':>8s}")
for c, sf, sb, adv, ret, t in R['race']:
    print(f'{c:5.1f}b{sf:+11.3f}{sb:+13.3f}{adv:+9.3f}{ret:+9.0f}bp{t:+8.2f}')
print(f"\nbootstrap excess-Sharpe CI: VEA [{R['ci_vea_lo']:+.3f}, {R['ci_vea_hi']:+.3f}]  "
      f"blend [{R['ci_blend_lo']:+.3f}, {R['ci_blend_hi']:+.3f}] -> heavily overlapping")
print('The "gross-yield" wrapper costs ~1 pp/yr and buys back no measurable tax.')

  cost   VEA exSh   blend exSh      adv    ret adv   HAC t
  0.0b     +0.288       +0.251   -0.037     -102bp   -1.05
  5.0b     +0.288       +0.250   -0.038     -103bp   -1.06
 25.0b     +0.288       +0.249   -0.040     -107bp   -1.09

bootstrap excess-Sharpe CI: VEA [-0.091, +0.702]  blend [-0.132, +0.652] -> heavily overlapping
The "gross-yield" wrapper costs ~1 pp/yr and buys back no measurable tax.


## Live synthetic control — the machinery is unbiased

A four-fund panel on one market factor. The broad fund loses an extra 8 pp of its dividend to withholding at `signal_strength=1` (a planted 25.6 bp/yr leak) and exactly the same as the benchmark at `signal_strength=0`. The estimator must recover the first and stay silent on the second.

In [9]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from withholding import data, strategy as st
pl = st.synthetic_detect(*data.synthetic_panel(signal_strength=1.0, seed=916), n_boot=400)
print('planted %.1f bp -> measured %+.1f bp (t %+.2f), CI [%+.1f, %+.1f]'
      % (pl['true_gap_bp'], pl['measured_gap_bp'], pl['t_hac'], pl['ci_low_bp'], pl['ci_high_bp']))
nulls = np.array([st.synthetic_detect(*data.synthetic_panel(signal_strength=0.0, seed=916+s),
                                     n_boot=200)['measured_gap_bp'] for s in range(5)])
print('null x5: mean %+.2f bp (sd %.2f), |gap|>=10 bp in %d/5'
      % (nulls.mean(), nulls.std(ddof=1), int((np.abs(nulls) >= 10).sum())))
half = st.synthetic_detect(*data.synthetic_panel(signal_strength=0.5, seed=916), n_boot=200)
print('dose response: half the planted leak -> %+.1f bp (half of %+.1f)'
      % (half['measured_gap_bp'], pl['measured_gap_bp']))

planted 25.6 bp -> measured +25.4 bp (t +9.17), CI [+24.6, +25.9]


null x5: mean -0.04 bp (sd 0.18), |gap|>=10 bp in 0/5


dose response: half the planted leak -> +12.7 bp (half of +25.4)


## Verdict

- **Signal — None.** The estimand is **unidentified on the tape**. Measurement is exact (max residual 0.60 bp against declared cash), but the gross dividend is never published in a price series and every US-listed benchmark suffers the same treaty withholding, so the constructed gap is fees plus composition. Headline: **-47.5 bp/yr** raw (HAC *t* = -1.07, wrong sign), **-0.5 bp** fee-adjusted (*t* = -0.01, CI [-69.3, +69.0]); insignificant in both eras with a sign flip between them; 72 bp of swing across country weights; and a resolution floor from the EFA/IEFA twin test that already fails to recover a *known* 26 bp fee gap. No |*t*| ≥ 2 in the right direction. The synthetic control recovers a planted 25.6 bp leak to +25.4 bp and is silent on the null (+0.1 bp), so the null is the tape's, not the harness's.
- **Tradability — Mirage.** Even granting the inferred 41 bp/yr central drag, no US-listed wrapper avoids it; the single-country benchmark costs 103 bp/yr more excess-of-cash (HAC *t* = -1.06) and is unaffected by cost assumptions. The only real lever is the foreign tax credit, which is a tax-return mechanic and off-tape.
- **Survivorship & proxies.** The seven funds are today's largest survivors (a mild ex-post tilt, named on the Signal axis). Expense ratios (a named anachronism — today's fees on a 19-year window, swept 35–55 bp), blend weights and the withholding rate are labelled assumptions and each is swept.